### Set up

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
from pathlib import Path
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)

<module 'src.translate_to_vis' from 'c:\\Users\\rylen\\Dropbox\\2025_moral_scenario_annotation\\code\\rylen\\src\\translate_to_vis.py'>

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [3]:
# set main paths

CUR_DIR = Path().resolve()
print(f"current_path: {CUR_DIR}")

SCENARIO_DIR = CUR_DIR / "scenarios_inputs" / "cheung_variants" 
OUTPUT_DIR = CUR_DIR / "annotated_outputs" / "cheung_variants"

current_path: C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen


## Look at Scenario

In [25]:
#set scenario file filename
FILENAME = 'tyrannicide.json'

#select scenario and action choice
SCENARIO_ID = 3 #looks for "id" entry, not index
ACT_ID = '1' #looks to "options" entry

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


You are an army officer in the Dominican Republic in 1961. For decades now, your country has been
governed by a dictator who has had thousands of people killed because of their ethnicity or their
political views. You are a member of a group that wants a new start for your country without a civil
war. To achieve this goal, you consider it unavoidable to take the dictator into custody in order to
disempower him, his family, and his followers. For this, you and six co-conspirators are asked to
take the dictator into custody. If you take the dictator into custody, the people in your country
will have a chance for a new start. If you do not take him into custody, the regime will continue
its killings. 




In [26]:
# print scenario json
print(json.dumps(scenario_json, indent=4))

{
    "id": 3,
    "scenario_title": "Tyrannicide",
    "deontology_level": "3",
    "utility_level": "3",
    "text": "You are an army officer in the Dominican Republic in 1961. For decades now, your country has been governed by a dictator who has had thousands of people killed because of their ethnicity or their political views. You are a member of a group that wants a new start for your country without a civil war. To achieve this goal, you consider it unavoidable to take the dictator into custody in order to disempower him, his family, and his followers. For this, you and six co-conspirators are asked to take the dictator into custody. If you take the dictator into custody, the people in your country will have a chance for a new start. If you do not take him into custody, the regime will continue its killings.",
    "options": {
        "1": "take the dictator into custody",
        "2": ""
    }
}


#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

#### Set up

In [27]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
# this_act_I = "I decide to " + this_act
this_act_I = this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv was instructed to take the dictator into custody.
take the dictator into custody
You are an army officer in the Dominican Republic in 1961. For decades now, your country has been governed by a dictator who has had thousands of people killed because of their ethnicity or their political views. You are a member of a group that wants a new start for your country without a civil war. To achieve this goal, you consider it unavoidable to take the dictator into custody in order to disempower him, his family, and his followers. For this, you and six co-conspirators are asked to take the dictator into custody. If you take the dictator into custody, the people in your country will have a chance for a new start. If you do not take him into custody, the regime will continue its killings.


Scenario:
Ziv is an army officer in the Dominican Republic in 1961. For decades now, their country has been governed by a dictator who has had thousands of people killed because of their eth

In [28]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')

Graph g initialized and reset.


#### Step 0. Get entities

In [29]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
the dictator
six co-conspirators


#### Step 1. Deontology / Action Value Scores

In [30]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
processed_values  = annotate_scenario.process_values_simple(this_scenario,this_act,this_act_I,g) 

print(processed_values)

    

You are an expert judge of the moral virtues of different actions. Moral actions are those which humans consider virtuous, that consider others' wellbeing and happiness, and that are guided by principles of ethics. You will be given a sentence describing an action. You must rate thus action in terms of its moral virtue. Respond with a rating from -100 to 100 for this action, where -100 is the most morally wrong action you can possibly think of, and 100 is the most morally virtuous action you can possibly think of. 0 indicates a morally neutral action. Please return a json object with key:value pair of "score": rating. Please rate this action: take the dictator into custody
{'score': 70}


#### Step 2. Outcome Likelihoods

In [31]:
#Step 2. Outcomes
processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

My co-conspirators and I detain the dictator
The dictator is disempowered
The dictator's family and followers lose power
The regime's killings decrease or stop
The country gains a chance for a new start
Potential political instability during transition
Possible public relief or support for my action
Risk of retaliation from loyalists to the dictator
I and my co-conspirators face personal risk
Improved safety for persecuted ethnic and political groups


#### Step 3. Outcome Utilities

In [32]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: My co-conspirators and I detain the dictator

Replacing dictator with: 
{'the dictator'}
Scored impacts for these beings:
['Ziv', 'six co-conspirators', 'the dictator']
Scored values:
[50, -100, 50]

Processing impacts of event: The dictator is disempowered

Replacing dictator with: 
{'the dictator'}
Scored impacts for these beings:
['Ziv', 'six co-conspirators', 'the dictator']
Scored values:
[70, -100, 60]

Processing impacts of event: The dictator's family and followers lose power

Replacing dictator with: 
{'the dictator'}
Scored impacts for these beings:
['Ziv', 'six co-conspirators', 'the dictator']
Scored values:
[50, -100, 50]

Processing impacts of event: The regime's killings decrease or stop

Replacing dictator with: 
{'the dictator'}
Scored impacts for these beings:
['Ziv', 'six co-conspirators', 'the dictator']
Scored values:
[80, -100, 80]

Processing impacts of event: The country gains a chance for a new start

Replacing dictator with: 
{'th

#### Step 4. Cause / Intend / Know Links

In [33]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: My co-conspirators and I detain the dictator
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The dictator is disempowered
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The dictator's family and followers lose power
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: The regime's killings decrease or stop
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-

Processing event: The country gains a chance for a new start
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-

Processing event: Potential political instability during transition
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+

Processing event: Possible public relief or support for my action
{'cause': 'yes', 'intend': 'yes', 'know': 'no'}
CKI links for I
C+I+K-

Processing event: Risk of retaliation from loyalists to the dicta

#### Step 5. Write out the results

In [34]:
#optional -- write out the results 

this_output_filename = f"{OUTPUT_DIR}/cheung_variants_tyrannicide_{scenario_json['id']}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen\annotated_outputs\cheung_variants/cheung_variants_tyrannicide_3_choice_1.json



C:\Users\rylen\Dropbox\2025_moral_scenario_annotation\code\rylen\annotated_outputs\cheung_variants/cheung_variants_tyrannicide_3_choice_1.json
